# USGS Site Classification: Regulated vs. Unregulated
### Using USGS Peak Flow Codes, GAGES-II, GRanD, NWIS Metadata for Quick Check


## Step 0 — Install & Import Libraries

In [18]:
import os, re, glob, requests, zipfile, io, warnings
import pandas as pd
import numpy as np
import dataretrieval.nwis as nwis
import geopandas as gpd
from shapely.geometry import box

warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


## Step 1 — Configuration: Your Site List

In [ ]:
# ---------------------------------------------------------------
# USER INPUT — folder containing your iv CSVs
# ---------------------------------------------------------------
DATA_FOLDER = r'Outputs\USGS_iv_data'
OUT_FOLDER  = r'Outputs'
# Create Classification subfolder
class_folder = os.path.join(OUT_FOLDER, 'Classification')
os.makedirs(class_folder, exist_ok=True)
os.makedirs(OUT_FOLDER, exist_ok=True)

station_file = os.path.join(OUT_FOLDER, "site_info.csv")
stations_df = pd.read_csv(station_file, dtype=str)
site_col    = stations_df.columns[0]   # auto-detects first column as site ID
# Pad USGS site IDs back to 8 digits with leading zeros
stations_df[site_col] = stations_df[site_col].str.strip().str.zfill(8)

site_ids = stations_df[site_col].str.strip().tolist()
print(f'Sites detected from folder: {len(site_ids)}')
print(site_ids)

Sites detected from folder: 101
['07148400', '07152000', '07153000', '07157950', '07158000', '07159100', '07159750', '07160000', '07160350', '07160500', '07161450', '07164600', '07165562', '07165565', '07165570', '07171000', '07174400', '07175500', '07176000', '07176500', '07177650', '07177800', '07185000', '07188000', '07189540', '07189542', '07191000', '07191220', '07191500', '07195855', '07195865', '07196000', '07196500', '07197000', '07197360', '07228500', '07229050', '07229200', '07229300', '07230000', '07230500', '07231000', '07231500', '07234000', '07237500', '07238000', '07239300', '07239450', '07239500', '07239700', '07240000', '07241000', '07241520', '07241550', '07242000', '07242380', '07243500', '07245000', '07247015', '07247250', '07247500', '07249413', '07249455', '07249800', '07249985', '07300500', '07301110', '07301420', '07303000', '07305000', '07307028', '07311000', '07311500', '07315700', '07316500', '07324200', '07324400', '07325000', '07325800', '07325850', '073258

---
## ✅ Method 1 — GAGES-II Classification (Primary)

**GAGES-II** classifies 9,322 CONUS USGS gages as either:
- `Ref` — **Reference (unregulated)**: least-disturbed watersheds
- `Non-ref` — **Non-reference (regulated/disturbed)**: affected by dams, diversions, or land use

The key column is **`CLASS`** in the `BasinID` table, downloaded directly from ScienceBase.

> Falcone, J.A., 2011, GAGES-II: Geospatial Attributes of Gages for Evaluating Streamflow,  
> ver. 2.0: U.S. Geological Survey Data Release. https://doi.org/10.3133/70046617

### 1a — Download GAGES-II `BasinID` Table

In [20]:
GAGESII_LOCAL = os.path.join(OUT_FOLDER, 'gagesII.csv')
df_gages = pd.read_csv(GAGESII_LOCAL, dtype={'STAID': str}, encoding='latin-1')

#add 0 to STAID if it is 7 digits
df_gages['STAID'] = df_gages['STAID'].apply(lambda x: x.zfill(8))

print(f'Loaded cached GAGES-II: {len(df_gages):,} gages')

print('\nColumns:', df_gages.columns.tolist()[:15], '...')
df_gages[['STAID','CLASS',]].head()

Loaded cached GAGES-II: 9,058 gages

Columns: ['STAID', 'CLASS', 'NDAMS_2009', 'STOR_NID_2009', 'AGGECOREGION', 'HYDRO_DISTURB_INDX', 'WR_REPORT_REMARKS', 'ADR_CITATION', 'SCREENING_COMMENTS'] ...


,STAID,CLASS
0,01011000,Non-ref
1,01013500,Ref
2,01015800,Non-ref
3,01016500,Non-ref
4,01017000,Non-ref


### 1b — Match Your Sites to GAGES-II

In [21]:
# Build lookup with your exact column names
gages_lookup = df_gages.set_index('STAID')[
    ['CLASS', 'AGGECOREGION', 'NDAMS_2009', 'STOR_NID_2009']
].to_dict('index')

def classify_regulation(gages_class, ndams, stor_nid):
    if gages_class == 'Ref':
        return 'Unregulated'
    elif float(ndams) > 0 and float(stor_nid) > 0.1:
        return 'Regulated'
    else:
        return 'Unregulated'

rows = []
for sid in site_ids:
    sid_pad = sid.zfill(8)
    if sid_pad in gages_lookup:
        info        = gages_lookup[sid_pad]
        gages_class = info['CLASS']
        ndams       = info['NDAMS_2009']    if info['NDAMS_2009']    not in [None, '', 'nan'] else 0
        stor_nid    = info['STOR_NID_2009'] if info['STOR_NID_2009'] not in [None, '', 'nan'] else 0
        regulation  = classify_regulation(gages_class, ndams, stor_nid)
        rows.append({
            'site_no'       : sid,
            'GAGESII_CLASS' : gages_class,
            'ecoregion'     : info['AGGECOREGION'],
            'NDAMS_2009'    : ndams,
            'STOR_NID_2009' : stor_nid,
            'method1_status': regulation,
            'in_GAGESII'    : True
        })
    else:
        rows.append({
            'site_no'       : sid,
            'GAGESII_CLASS' : None,
            'ecoregion'     : None,
            'NDAMS_2009'    : None,
            'STOR_NID_2009' : None,
            'method1_status': 'Not in GAGES-II',
            'in_GAGESII'    : False
        })

df_m1 = pd.DataFrame(rows)
print('Method 1 classification summary:')
print(df_m1['method1_status'].value_counts().to_string())
print()
display(df_m1)

Method 1 classification summary:
method1_status
Regulated          77
Unregulated        15
Not in GAGES-II     9



,site_no,GAGESII_CLASS,ecoregion,NDAMS_2009,STOR_NID_2009,method1_status,in_GAGESII
0,07148400,Ref,WestPlains,60.0,3.80,Unregulated,True
1,07152000,Non-ref,WestPlains,30.0,4.06,Regulated,True
2,07153000,Non-ref,WestPlains,108.0,98.20,Regulated,True
3,07157950,Non-ref,WestPlains,110.0,1.83,Regulated,True
4,07158000,Non-ref,WestPlains,207.0,2.13,Regulated,True
...,...,...,...,...,...,...,...
96,07336200,Non-ref,EastHghlnds,18.0,526.30,Regulated,True
97,07337900,Non-ref,EastHghlnds,1.0,0.14,Regulated,True
98,07338500,Non-ref,EastHghlnds,7.0,375.46,Regulated,True
99,07338750,Non-ref,EastHghlnds,6.0,5.01,Regulated,True


---
## 🔶 Method 2 — GRanD Dam Database (Supplementary)

Uses the **Global Reservoir and Dam Database (GRanD v1.3)** to check if any large dam  
exists **upstream within the same HUC8 watershed** of each site.  
Sites with an upstream GRanD dam are flagged as regulated.

> **Requires:** `geopandas`, `shapely`  
> Best used for sites **not found in GAGES-II** or as a cross-check.

> Lehner, B. et al. (2011). High-resolution mapping of the world's reservoirs and dams.  
> Frontiers in Ecology and the Environment.

In [22]:
RUN_METHOD2  = False           # ← Set True to activate
GRAND_SHP    = r'Outputs\GRanD\GDW_reservoirs_v1_0.shp'
SEARCH_DEG   = 1.0            # degrees (~110 km) bounding box around each site

if RUN_METHOD2:
    # Load GRanD
    gdf_grand = gpd.read_file(GRAND_SHP)[['GRAND_ID', 'DAM_NAME', 'RES_NAME',
                                           'COUNTRY', 'CAP_MCM', 'geometry']]
    # Replace geometry with centroid so .x always works regardless of geometry type
    gdf_grand['geometry'] = gdf_grand['geometry'].centroid
    print(f'GRanD loaded: {len(gdf_grand):,} dams')

    # Use lat/lon already in station_df — no NWIS call needed
    # Expects columns: site_no, lat, lon
    site_info = stations_df[['site_no', 'lat', 'lon']].copy()
    site_info['site_no'] = site_info['site_no'].astype(str).str.zfill(8)

    # ── Fix: convert lat/lon to numeric ──────────────────────────────────
    site_info['lat'] = pd.to_numeric(site_info['lat'], errors='coerce')
    site_info['lon'] = pd.to_numeric(site_info['lon'], errors='coerce')

    m2_rows = []
    for _, row in site_info.iterrows():
        lat, lon = row['lat'], row['lon']

        if pd.isna(lat) or pd.isna(lon):
            m2_rows.append({
                'site_no'       : row['site_no'],
                'method2_status': 'No coords',
                'grand_dam'     : None,
                'grand_cap_mcm' : None
            })
            continue

        # Bounding box search around site
        bbox   = box(lon - SEARCH_DEG, lat - SEARCH_DEG,
                     lon + SEARCH_DEG, lat + SEARCH_DEG)
        nearby = gdf_grand[gdf_grand.geometry.within(bbox)]

        # Only flag as regulated if dam is roughly upstream (lon ≤ site lon)
        upstream = nearby[nearby.geometry.x <= lon]

        if len(upstream) > 0:
            dam_names = ', '.join(upstream['DAM_NAME'].dropna().unique()[:3])
            total_cap = upstream['CAP_MCM'].dropna().sum()
            m2_rows.append({
                'site_no'       : row['site_no'],
                'method2_status': 'Regulated',
                'grand_dam'     : dam_names,
                'grand_cap_mcm' : round(total_cap, 2)
                })
        else:
           m2_rows.append({
                'site_no'       : row['site_no'],
                'method2_status': 'Unregulated',
                'grand_dam'     : None,
                'grand_cap_mcm' : None
            })

    df_m2 = pd.DataFrame(m2_rows)
    print('\nMethod 2 summary:')
    print(df_m2['method2_status'].value_counts().to_string())
    display(df_m2)

else:
    print('Method 2 skipped (RUN_METHOD2 = False).')
    df_m2 = pd.DataFrame({
        'site_no'       : pd.Series(site_ids).str.zfill(8),
        'method2_status': 'Skipped',
        'grand_dam'     : None,
        'grand_cap_mcm' : None
    })

Method 2 skipped (RUN_METHOD2 = False).


## 🔷 Method 3 — NWIS Peak Flow Classification

In [23]:
# ── METHOD 3 — Peak-Flow Qualification Code Classification ─────────────

RUN_METHOD3  = True
PEAK_FOLDER  = r'Outputs\USGS_peak_data'

# ── Code definitions ───────────────────────────────────────────────────
# Code 6 = Regulated or Affected by Regulation
# Code 5 = Affected by Regulation or Diversion (likely regulated)
CODE_REGULATED        = '6'
CODE_LIKELY_REGULATED = '5'

def parse_peak_codes(raw):
    """
    Extract only valid single-digit integer codes from a peak_cd string.
    Handles values like '6', '5 6', 'C6', '6A', nan, '', etc.
    Returns a set of digit strings found (e.g. {'5', '6'}).
    """
    if pd.isna(raw) or str(raw).strip() == '':
        return set()
    # Keep only digit characters, split into individual digits
    digits = set(re.findall(r'\d', str(raw)))
    return digits

def classify_peak_record(raw_cd):
    """
    Classify a single peak record based on parsed codes.
    Priority: Regulated (6) > Likely Regulated (5) > Unregulated
    """
    codes = parse_peak_codes(raw_cd)
    if CODE_REGULATED in codes:
        return 'Regulated'
    elif CODE_LIKELY_REGULATED in codes:
        return 'Likely Regulated'
    else:
        return 'Unregulated'

if RUN_METHOD3:

    csv_files = glob.glob(os.path.join(PEAK_FOLDER, '*_USGS_peak.csv'))
    print(f'Peak-flow files found: {len(csv_files)}')

    frames = []
    for fpath in csv_files:
        fname = os.path.basename(fpath)
        if not re.match(r'^\d{8}', fname):
            continue
        df_tmp = pd.read_csv(fpath, dtype={'USGS_site': str, 'peak_cd': str})
        frames.append(df_tmp)

    if not frames:
        print('No peak-flow files loaded. Check PEAK_FOLDER path.')
    else:
        df_peak = pd.concat(frames, ignore_index=True)
        df_peak['USGS_site'] = df_peak['USGS_site'].astype(str).str.zfill(8)

        # ── Classify each individual peak record ──────────────────────
        df_peak['parsed_codes']   = df_peak['peak_cd'].apply(parse_peak_codes)
        df_peak['record_status']  = df_peak['peak_cd'].apply(classify_peak_record)

        # ── Aggregate to site level ────────────────────────────────────
        m3_rows = []
        for site_no, grp in df_peak.groupby('USGS_site'):

            total_peaks     = len(grp)
            reg_grp         = grp[grp['record_status'] == 'Regulated']
            likely_reg_grp  = grp[grp['record_status'] == 'Likely Regulated']
            n_reg           = len(reg_grp)
            n_likely        = len(likely_reg_grp)

            def years_from_grp(g, max_show=5):
                """Return comma-separated list of unique years (up to max_show)."""
                return ', '.join(
                    g['peak_date'].dropna().astype(str).str[:4]
                    .unique()[:max_show]
                )

            def unique_raw_codes(g):
                """Return unique non-null raw peak_cd strings observed."""
                return ', '.join(
                    g['peak_cd'].dropna()
                    .astype(str)
                    .str.strip()
                    .loc[lambda s: s != '']
                    .unique()
                )

            if n_reg > 0:
                site_status = 'Regulated'
            elif n_likely > 0:
                site_status = 'Likely Regulated'
            else:
                site_status = 'Unregulated'

            m3_rows.append({
                'site_no'               : site_no,
                'method3_status'        : site_status,
                # Raw peak_cd values actually seen for flagged records
                'peak_cd_regulated'     : unique_raw_codes(reg_grp)        or None,
                'peak_cd_likely'        : unique_raw_codes(likely_reg_grp) or None,
                # Years carrying each flag
                'years_regulated'       : years_from_grp(reg_grp)          or None,
                'years_likely_regulated': years_from_grp(likely_reg_grp)   or None,
                # Counts
                'n_regulated'           : n_reg,
                'n_likely_regulated'    : n_likely,
                'n_unregulated'         : total_peaks - n_reg - n_likely,
                'n_total_peaks'         : total_peaks,
                'pct_regulated'         : round(n_reg    / total_peaks * 100, 1),
                'pct_likely_regulated'  : round(n_likely / total_peaks * 100, 1),
            })

        df_m3 = pd.DataFrame(m3_rows)

        # ── Summary ───────────────────────────────────────────────────
        print('\nMethod 3 — Site-level classification summary:')
        print(df_m3['method3_status'].value_counts().to_string())
        print(f'\nTotal sites : {len(df_m3)}')
        print(f'  Regulated         : {(df_m3["method3_status"]=="Regulated").sum()}')
        print(f'  Likely Regulated  : {(df_m3["method3_status"]=="Likely Regulated").sum()}')
        print(f'  Unregulated       : {(df_m3["method3_status"]=="Unregulated").sum()}')

        display(df_m3)

        # ── Save ──────────────────────────────────────────────────────
        out_m3 = os.path.join(class_folder, 'method3_regulation_status.csv')
        df_m3.to_csv(out_m3, index=False)
        print(f'\nMethod 3 results saved → {out_m3}')

else:
    print('Method 3 skipped (RUN_METHOD3 = False).')
    df_m3 = pd.DataFrame({
        'site_no'               : pd.Series(site_ids).str.zfill(8),
        'method3_status'        : 'Skipped',
        'peak_cd_regulated'     : None,
        'peak_cd_likely'        : None,
        'years_regulated'       : None,
        'years_likely_regulated': None,
        'n_regulated'           : None,
        'n_likely_regulated'    : None,
        'n_unregulated'         : None,
        'n_total_peaks'         : None,
        'pct_regulated'         : None,
        'pct_likely_regulated'  : None,
    })

Peak-flow files found: 101

Method 3 — Site-level classification summary:
method3_status
Likely Regulated    52
Regulated           34
Unregulated         15

Total sites : 101
  Regulated         : 34
  Likely Regulated  : 52
  Unregulated       : 15


,site_no,method3_status,peak_cd_regulated,peak_cd_likely,years_regulated,years_likely_regulated,n_regulated,n_likely_regulated,n_unregulated,n_total_peaks,pct_regulated,pct_likely_regulated
0,07148400,Unregulated,None,None,None,None,0,0,60,60,0.0,0.0
1,07152000,Likely Regulated,None,"5, 3,5",None,"1978, 1979, 1981, 1982, 1983",0,46,43,89,0.0,51.7
2,07153000,Likely Regulated,None,5,None,"1968, 1969, 1970, 1971, 1973",0,58,25,83,0.0,69.9
3,07157950,Likely Regulated,None,5,None,"1978, 1979, 1981, 1982, 1983",0,41,18,59,0.0,69.5
4,07158000,Likely Regulated,None,5,None,"1978, 1979, 1981, 1982, 1983",0,47,40,87,0.0,54.0
...,...,...,...,...,...,...,...,...,...,...,...,...
96,07336200,Likely Regulated,None,"5, 1,2,5, 2,5",None,"1973, 1974, 1976, 1977, 1978",0,53,0,53,0.0,100.0
97,07337900,Unregulated,None,None,None,None,0,0,65,65,0.0,0.0
98,07338500,Regulated,6,None,"1969, 1970, 1971, 1974, 1976",None,56,0,39,95,58.9,0.0
99,07338750,Unregulated,None,None,None,None,0,0,32,32,0.0,0.0



Method 3 results saved → Outputs\Classification\method3_regulation_status.csv


---
### 🔷 Method 4 — NWIS Site Metadata (Quick Check)

Uses `dataretrieval.nwis.get_info()` to retrieve site-level metadata from NWIS.  
The **`alterations_cd`** field (when present) indicates whether a site is affected by regulation.  
Additionally, **`contrib_drain_area_va`** vs **`drain_area_va`** discrepancies can indicate  
upstream storage effects.

> ⚡ **No file download needed** — queries NWIS directly via API.  
> Best as a fast sanity check on top of Method 1.

In [24]:
# Fetch NWIS site info for all your sites
print(f'Querying NWIS metadata for {len(site_ids)} sites ...')
site_meta, _ = nwis.get_info(sites=site_ids)
site_meta = site_meta.reset_index()

# Show all columns so you can see what metadata is available
print('\nAvailable metadata columns:')
print(site_meta.columns.tolist())
site_meta.head(3)

Querying NWIS metadata for 101 sites ...

Available metadata columns:
['index', 'agency_cd', 'site_no', 'station_nm', 'site_tp_cd', 'lat_va', 'long_va', 'dec_lat_va', 'dec_long_va', 'coord_meth_cd', 'coord_acy_cd', 'coord_datum_cd', 'dec_coord_datum_cd', 'district_cd', 'state_cd', 'county_cd', 'country_cd', 'land_net_ds', 'map_nm', 'map_scale_fc', 'alt_va', 'alt_meth_cd', 'alt_acy_va', 'alt_datum_cd', 'huc_cd', 'basin_cd', 'topo_cd', 'instruments_cd', 'construction_dt', 'inventory_dt', 'drain_area_va', 'contrib_drain_area_va', 'tz_cd', 'local_time_fg', 'reliability_cd', 'gw_file_cd', 'nat_aqfr_cd', 'aqfr_cd', 'aqfr_type_cd', 'well_depth_va', 'hole_depth_va', 'depth_src_cd', 'project_no', 'geometry']


,index,agency_cd,site_no,station_nm,site_tp_cd,lat_va,long_va,dec_lat_va,dec_long_va,coord_meth_cd,...,reliability_cd,gw_file_cd,nat_aqfr_cd,aqfr_cd,aqfr_type_cd,well_depth_va,hole_depth_va,depth_src_cd,project_no,geometry
0,0,USGS,07148400,"Salt Fork Arkansas River nr Alva, OK",ST,364854.0,983852.0,36.815031,-98.648139,M,...,NaN,NYNNNNNN,NaN,NaN,NaN,NaN,NaN,NaN,464000100,POINT (-98.64814 36.81503)
1,1,USGS,07152000,"Chikaskia River near Blackwell, OK",ST,364841.0,971637.0,36.811421,-97.277265,M,...,NaN,NYNNNNNN,NaN,NaN,NaN,NaN,NaN,NaN,464000100,POINT (-97.27726 36.81142)
2,2,USGS,07153000,"Black Bear Creek at Pawnee, OK",ST,362037.0,964757.0,36.343665,-96.799479,M,...,NaN,NYNNNNNN,NaN,NaN,NaN,NaN,NaN,NaN,464000100,POINT (-96.79948 36.34367)


In [25]:
# Key columns to inspect for regulation clues
CHECK_COLS = ['site_no','station_nm','state_cd','drain_area_va',
              'contrib_drain_area_va','nat_aqfr_cd','site_tp_cd']

available = [c for c in CHECK_COLS if c in site_meta.columns]
df_m4_meta = site_meta[available].copy()
df_m4_meta['site_no'] = df_m4_meta['site_no'].astype(str).str.zfill(8)

# Drainage area ratio: if contrib << drain, upstream storage likely exists
# (ratio < 0.85 is a common threshold used in literature)
if 'drain_area_va' in df_m4_meta.columns and 'contrib_drain_area_va' in df_m4_meta.columns:
    df_m4_meta['drain_ratio'] = (
        pd.to_numeric(df_m4_meta['contrib_drain_area_va'], errors='coerce') /
        pd.to_numeric(df_m4_meta['drain_area_va'], errors='coerce')
    )
    df_m4_meta['method4_status'] = df_m4_meta['drain_ratio'].apply(
        lambda r: 'Likely Regulated' if (pd.notna(r) and r < 0.85) else 'Likely Unregulated'
    )
else:
    df_m4_meta['drain_ratio']    = None
    df_m4_meta['method4_status'] = 'Insufficient metadata'

print('Method 4 summary:')
print(df_m4_meta['method4_status'].value_counts().to_string())
display(df_m4_meta)

Method 4 summary:
method4_status
Likely Unregulated    78
Likely Regulated      23


,site_no,station_nm,state_cd,drain_area_va,contrib_drain_area_va,nat_aqfr_cd,site_tp_cd,drain_ratio,method4_status
0,07148400,"Salt Fork Arkansas River nr Alva, OK",40,982.0,982.0,NaN,ST,1.000000,Likely Unregulated
1,07152000,"Chikaskia River near Blackwell, OK",40,1876.0,1873.0,NaN,ST,0.998401,Likely Unregulated
2,07153000,"Black Bear Creek at Pawnee, OK",40,538.0,538.0,NaN,ST,1.000000,Likely Unregulated
3,07157950,"Cimarron River near Buffalo, OK",40,12205.0,8177.0,NaN,ST,0.669971,Likely Regulated
4,07158000,"Cimarron River near Waynoka, OK",40,13399.0,9371.0,NaN,ST,0.699381,Likely Regulated
...,...,...,...,...,...,...,...,...,...
96,07336200,"Kiamichi River near Antlers, OK",40,1129.0,1129.0,NaN,ST,1.000000,Likely Unregulated
97,07337900,"Glover River near Glover, OK",40,320.0,320.0,NaN,ST,1.000000,Likely Unregulated
98,07338500,"Little River blw Lukfata Creek, nr Idabel, OK",40,1228.0,1228.0,NaN,ST,1.000000,Likely Unregulated
99,07338750,"Mountain Fork at Smithville, OK",40,322.0,322.0,NaN,ST,1.000000,Likely Unregulated


---
## Step 4 — Merge All Methods → Final Classification

In [29]:
# ── Classification rules ───────────────────────────────────────────────
#
#  Rule 1 : M1 & M3 agree (both Regulated OR both Unregulated)
#           → agreed label
#
#  Rule 2 : M1 = Unregulated  AND  M3 = Likely Regulated
#           → Unregulated  (M1 overrides weak M3 signal)
#
#  Rule 3 : Not in M1  AND  M3 = Regulated (strictly)
#           → Regulated
#
#  Rule 4 : Not in M1  AND  M3 = Regulated OR Likely Regulated
#                       AND  M4 = Likely Regulated
#           → Regulated
#
#  Default: Unregulated
# ──────────────────────────────────────────────────────────────────────

# ── Merge all method results ───────────────────────────────────────────
df_final = df_m1.copy()
df_final = df_final.merge(df_m2[['site_no', 'method2_status', 'grand_dam', 'grand_cap_mcm']], 
                           on='site_no', how='left')
df_final = df_final.merge(df_m3[['site_no', 'method3_status', 'peak_cd_regulated', 
                                   'peak_cd_likely', 'years_regulated', 'years_likely_regulated',
                                   'n_regulated', 'n_likely_regulated', 'n_unregulated', 'n_total_peaks',
                                   'pct_regulated', 'pct_likely_regulated']], 
                           on='site_no', how='left')
df_final = df_final.merge(df_m4_meta[['site_no', 'method4_status', 'drain_ratio']], 
                           on='site_no', how='left')

# ── Helper function: clean null values ─────────────────────────────────
def clean(val):
    """Return None if val is null/NaN, else return the value."""
    return None if pd.isna(val) else val

print(f'Merged dataframe shape: {df_final.shape}')
print(f'Columns: {df_final.columns.tolist()}')

# ── Now run your classification functions ──────────────────────────────
def final_classify(row):
    m1_raw = clean(row.get('method1_status'))
    m3_raw = clean(row.get('method3_status'))
    m4_raw = clean(row.get('method4_status'))

    in_m1  = bool(row.get('in_GAGESII', False)) and m1_raw is not None

    # Core labels (strip 'Likely ' for Rule 1 agreement check only)
    m1_core = m1_raw.replace('Likely ', '') if m1_raw else None
    m3_core = m3_raw.replace('Likely ', '') if m3_raw else None

    # ── Rule 1: M1 & M3 agree ─────────────────────────────────────────
    if in_m1 and m3_core is not None and m1_core == m3_core:
        return m1_core                          # 'Regulated' or 'Unregulated'

    # ── Rule 2: M1 = Unregulated, M3 = Likely Regulated ──────────────
    if in_m1 and m1_core == 'Unregulated' and m3_raw == 'Likely Regulated':
        return 'Unregulated'

    # ── Rules 3 & 4 apply only when site is NOT in Method 1 ───────────
    if not in_m1:

        # Rule 3: M3 strictly Regulated (no M4 required)
        if m3_raw == 'Regulated':
            return 'Regulated'

        # Rule 4: M3 Regulated or Likely Regulated  +  M4 Likely Regulated
        m3_is_reg    = m3_raw in ('Regulated', 'Likely Regulated')
        m4_is_likely = m4_raw in ('Regulated', 'Likely Regulated')
        if m3_is_reg and m4_is_likely:
            return 'Regulated'

    # ── Default ───────────────────────────────────────────────────────
    return 'Unregulated'


def classify_reason(row):
    m1_raw = clean(row.get('method1_status'))
    m3_raw = clean(row.get('method3_status'))
    m4_raw = clean(row.get('method4_status'))

    in_m1  = bool(row.get('in_GAGESII', False)) and m1_raw is not None
    m1_core = m1_raw.replace('Likely ', '') if m1_raw else None
    m3_core = m3_raw.replace('Likely ', '') if m3_raw else None

    if in_m1 and m3_core is not None and m1_core == m3_core:
        return f'Rule 1 — M1 & M3 agree ({m1_core})'

    if in_m1 and m1_core == 'Unregulated' and m3_raw == 'Likely Regulated':
        return 'Rule 2 — M1=Unregulated overrides M3=Likely Regulated → Unregulated'

    if not in_m1:
        if m3_raw == 'Regulated':
            return 'Rule 3 — Not in M1; M3=Regulated → Regulated'

        m3_is_reg    = m3_raw in ('Regulated', 'Likely Regulated')
        m4_is_likely = m4_raw in ('Regulated', 'Likely Regulated')
        if m3_is_reg and m4_is_likely:
            return f'Rule 4 — Not in M1; M3={m3_raw} + M4={m4_raw} → Regulated'

    return 'Default → Unregulated'


df_final['final_status']          = df_final.apply(final_classify,  axis=1)
df_final['classification_reason'] = df_final.apply(classify_reason, axis=1)

# ── Summary ───────────────────────────────────────────────────────────
print('=== FINAL CLASSIFICATION SUMMARY ===')
print(df_final['final_status'].value_counts().to_string())
print()
print('=== RULE BREAKDOWN ===')
print(df_final['classification_reason'].value_counts().to_string())
print()

display(df_final[[
    'site_no',
    'GAGESII_CLASS',
    'method1_status',
    'method2_status',
    'method3_status',
    'peak_cd_regulated',
    'peak_cd_likely',
    'method4_status',
    'drain_ratio',
    'final_status',
    'classification_reason'
]])

out_final = os.path.join(class_folder, 'final_regulation_classification.csv')
df_final.to_csv(out_final, index=False)
print(f'\nFinal classification saved → {out_final}')

Merged dataframe shape: (101, 23)
Columns: ['site_no', 'GAGESII_CLASS', 'ecoregion', 'NDAMS_2009', 'STOR_NID_2009', 'method1_status', 'in_GAGESII', 'method2_status', 'grand_dam', 'grand_cap_mcm', 'method3_status', 'peak_cd_regulated', 'peak_cd_likely', 'years_regulated', 'years_likely_regulated', 'n_regulated', 'n_likely_regulated', 'n_unregulated', 'n_total_peaks', 'pct_regulated', 'pct_likely_regulated', 'method4_status', 'drain_ratio']
=== FINAL CLASSIFICATION SUMMARY ===
final_status
Regulated      79
Unregulated    22

=== RULE BREAKDOWN ===
classification_reason
Rule 1 — M1 & M3 agree (Regulated)                                           72
Rule 1 — M1 & M3 agree (Unregulated)                                          9
Default → Unregulated                                                         7
Rule 2 — M1=Unregulated overrides M3=Likely Regulated → Unregulated           6
Rule 3 — Not in M1; M3=Regulated → Regulated                                  5
Rule 4 — Not in M1; M3=Li

,site_no,GAGESII_CLASS,method1_status,method2_status,method3_status,peak_cd_regulated,peak_cd_likely,method4_status,drain_ratio,final_status,classification_reason
0,07148400,Ref,Unregulated,Skipped,Unregulated,None,None,Likely Unregulated,1.000000,Unregulated,Rule 1 — M1 & M3 agree (Unregulated)
1,07152000,Non-ref,Regulated,Skipped,Likely Regulated,None,"5, 3,5",Likely Unregulated,0.998401,Regulated,Rule 1 — M1 & M3 agree (Regulated)
2,07153000,Non-ref,Regulated,Skipped,Likely Regulated,None,5,Likely Unregulated,1.000000,Regulated,Rule 1 — M1 & M3 agree (Regulated)
3,07157950,Non-ref,Regulated,Skipped,Likely Regulated,None,5,Likely Regulated,0.669971,Regulated,Rule 1 — M1 & M3 agree (Regulated)
4,07158000,Non-ref,Regulated,Skipped,Likely Regulated,None,5,Likely Regulated,0.699381,Regulated,Rule 1 — M1 & M3 agree (Regulated)
...,...,...,...,...,...,...,...,...,...,...,...
96,07336200,Non-ref,Regulated,Skipped,Likely Regulated,None,"5, 1,2,5, 2,5",Likely Unregulated,1.000000,Regulated,Rule 1 — M1 & M3 agree (Regulated)
97,07337900,Non-ref,Regulated,Skipped,Unregulated,None,None,Likely Unregulated,1.000000,Unregulated,Default → Unregulated
98,07338500,Non-ref,Regulated,Skipped,Regulated,6,None,Likely Unregulated,1.000000,Regulated,Rule 1 — M1 & M3 agree (Regulated)
99,07338750,Non-ref,Regulated,Skipped,Unregulated,None,None,Likely Unregulated,1.000000,Unregulated,Default → Unregulated



Final classification saved → Outputs\Classification\final_regulation_classification.csv


## Step 5 — Export Results

In [39]:
# Full classification table
NWM_FOLDER = r'Outputs\NWM_ins_data'
full_path = os.path.join(class_folder, 'site_classification_all_methods.csv')
df_final.to_csv(full_path, index=False)
print(f'Saved: {full_path}')

# Separate regulated / unregulated site lists
unreg = df_final[df_final['final_status'] == 'Unregulated']
reg   = df_final[df_final['final_status'] == 'Regulated']

reg_folder = os.path.join(class_folder, 'Regulated')
unreg_folder = os.path.join(class_folder, 'Unregulated')
os.makedirs(reg_folder, exist_ok=True)
os.makedirs(unreg_folder, exist_ok=True)

unreg[['site_no','GAGESII_CLASS','final_status']].to_csv(os.path.join(unreg_folder, 'sites_unregulated.csv'), index=False)
reg[['site_no','GAGESII_CLASS','final_status']].to_csv(os.path.join(reg_folder, 'sites_regulated.csv'), index=False)

print(f'Saved: {os.path.join(unreg_folder, "sites_unregulated.csv")}  ({len(unreg)} sites)')
print(f'Saved: {os.path.join(reg_folder, "sites_regulated.csv")}    ({len(reg)} sites)')

Saved: Outputs\Classification\site_classification_all_methods.csv
Saved: Outputs\Classification\Unregulated\sites_unregulated.csv  (22 sites)
Saved: Outputs\Classification\Regulated\sites_regulated.csv    (79 sites)


In [41]:
# Copy actual IV data files to classified folders
import shutil

# Separate regulated / unregulated site lists
unreg = df_final[df_final['final_status'] == 'Unregulated']
reg   = df_final[df_final['final_status'] == 'Regulated']

# Get lists of site IDs for regulated and unregulated
reg_sites = reg['site_no'].tolist()
unreg_sites = unreg['site_no'].tolist()

# Copy regulated files
for site_no in reg_sites:
    # Copy USGS file
    src_usgs = os.path.join(DATA_FOLDER, f'{site_no}_USGS_inst.csv')
    if os.path.exists(src_usgs):
        shutil.copy(src_usgs, reg_folder)
        print(f'Copied {site_no}_USGS_inst.csv to Regulated folder')
    else:
        print(f'Warning: {src_usgs} not found')
    
    # Copy NWM file
    src_nwm = os.path.join(NWM_FOLDER, f'{site_no}_NWM_inst.csv')
    if os.path.exists(src_nwm):
        shutil.copy(src_nwm, reg_folder)
        print(f'Copied {site_no}_NWM_inst.csv to Regulated folder')
    else:
        print(f'Warning: {src_nwm} not found')

# Copy unregulated files
for site_no in unreg_sites:
    # Copy USGS file
    src_usgs = os.path.join(DATA_FOLDER, f'{site_no}_USGS_inst.csv')
    if os.path.exists(src_usgs):
        shutil.copy(src_usgs, unreg_folder)
        print(f'Copied {site_no}_USGS_inst.csv to Unregulated folder')
    else:
        print(f'Warning: {src_usgs} not found')
    
    # Copy NWM file
    src_nwm = os.path.join(NWM_FOLDER, f'{site_no}_NWM_inst.csv')
    if os.path.exists(src_nwm):
        shutil.copy(src_nwm, unreg_folder)
        print(f'Copied {site_no}_NWM_inst.csv to Unregulated folder')
    else:
        print(f'Warning: {src_nwm} not found')

print(f'\nCopied data files to respective folders.')

Copied 07152000_USGS_inst.csv to Regulated folder
Copied 07152000_NWM_inst.csv to Regulated folder
Copied 07153000_USGS_inst.csv to Regulated folder
Copied 07153000_NWM_inst.csv to Regulated folder
Copied 07157950_USGS_inst.csv to Regulated folder
Copied 07157950_NWM_inst.csv to Regulated folder
Copied 07158000_USGS_inst.csv to Regulated folder
Copied 07158000_NWM_inst.csv to Regulated folder
Copied 07159100_USGS_inst.csv to Regulated folder
Copied 07159100_NWM_inst.csv to Regulated folder
Copied 07159750_USGS_inst.csv to Regulated folder
Copied 07159750_NWM_inst.csv to Regulated folder
Copied 07160000_USGS_inst.csv to Regulated folder
Copied 07160000_NWM_inst.csv to Regulated folder
Copied 07160350_USGS_inst.csv to Regulated folder
Copied 07160350_NWM_inst.csv to Regulated folder
Copied 07160500_USGS_inst.csv to Regulated folder
Copied 07160500_NWM_inst.csv to Regulated folder
Copied 07161450_USGS_inst.csv to Regulated folder
Copied 07161450_NWM_inst.csv to Regulated folder
Copied 071